# 06 - CI Gate For Data And AI Changes

This notebook models Metatate as a release gate. A proposed SQL model, export job, or AI workflow change is checked before it ships.

The notebook uses the same `cicd_policy_gate` package as the command-line runner, so the walkthrough and the CI job exercise the same implementation.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Load a pull request change set


In [ ]:
from cicd_policy_gate import DEFAULT_CHANGESET_PATH, evaluate_changes, load_changes
from cicd_policy_gate.gate import print_summary

change_set = load_changes(DEFAULT_CHANGESET_PATH)
print(f"{change_set['change_set_id']}: {change_set['description']}")
pd.DataFrame(change_set["changes"])[
    ["change_id", "kind", "source_path", "description"]
]


## Evaluate each change through Metatate


In [ ]:
strict = os.getenv("METATATE_EXAMPLES_STRICT_CI_GATE") == "1"
fail_on_controls = os.getenv("METATATE_EXAMPLES_FAIL_ON_CONTROLS") == "1"

summary = evaluate_changes(
    client,
    change_set,
    strict=strict,
    fail_on_controls=fail_on_controls,
)
gate_results = pd.DataFrame(summary.to_dict()["results"])
gate_results[
    ["change_id", "kind", "decision", "gate", "evidence_id", "rationale"]
]


In [ ]:
print_summary(summary)

if strict and not summary.release_allowed:
    raise AssertionError("Release gate failed. Resolve denied changes before deployment.")

if not strict:
    print("\nStrict mode is off. Set METATATE_EXAMPLES_STRICT_CI_GATE=1 to make this notebook fail like CI.")


## Machine-readable CI report


In [ ]:
report = summary.to_dict()
print(json.dumps(report, indent=2))


The same implementation can run outside notebooks:

```bash
scripts/run_cicd_policy_gate.sh --strict
```


In a real pipeline, denied changes stop the deployment. Conditional changes can create approval tasks, require anonymization, or fail the gate when `--fail-on-controls` is enabled.
